# Public preparation — read before running
Academic work by **Malika Fofana & Klara Mladenovic**.
Prepared with AI-assisted engineering. Outputs removed; no medical images redistributed.
Historical experiments used an image-level split. Patient independence is unverified.
Paths and explicit OpenCV import were repaired; model training has NOT been rerun.
ResNet input preprocessing remains as in the historical experiment: review AUDIT.md before interpreting results.
Use audit_dataset.py to audit a licensed dataset; group split is a separate proposed protocol, not the historical protocol.


# Projet Final: Classification d'IRM cérébrales pour la détection du déclin cognitif lié à Alzheimer

---

| | |
|:---|:---|
| **Étudiantes** | Malika Fofana & Klara Mladenovic |
| **Promotion** | PGE3 Fr |
| **Dataset** | Alzheimer MRI Dataset : 4 classes de sévérité |
| **Date** | Mars 2026 |

---

> *« Le vieillissement touche tout être humain. La santé ressemble à une descente de gradient :*
> *dans tous les cas elle évolue, mais la vitesse et la direction dépendent de comment on la traite.*
> *Notre ambition est d'utiliser l'IA non pour remplacer le regard clinique,*
> *mais pour l'assister  plus tôt, plus précisément, plus personnellement. »*

---

### Plan du notebook

| # | Section | Critère du sujet |
|---|---|---|
| 1 | Introduction & contexte médical | Introduction |
| 2 | Imports & configuration | — |
| 3 | Présentation du dataset | Présentation dataset |
| 4 | Prétraitement & split | Prétraitement |
| 5 | Analyse exploratoire | EDA |
| 6 | Modèle Baseline | Baseline |
| 7 | CNN Amélioré | CNN amélioré |
| 8 | Transfer Learning : ResNet50 | Transfer Learning |
| 9 | Évaluation comparative | Évaluation |
| 10 | Interprétabilité : Grad-CAM | Optionnel |
| 11 | Augmentation par GAN (DCGAN) | Option GAN |
| 12 | Réentraînement après GAN | Option GAN |
| 13 | Discussion critique | Discussion |
| 14 | Conclusion & perspectives | Conclusion |
| 15 | Références | — |


---
## 1. Introduction & Contexte médical

### 1.1 Notre angle : le vieillissement comme trajectoire

En data science, on parle souvent d'optimisation, de convergence, de fonctions de coût.
Mais ce projet nous rappelle que derrière les données, il y a des trajectoires humaines.

Le vieillissement cérébral n'est pas uniforme. Deux personnes du même âge peuvent avoir des cerveaux
très différents  l'une garde toutes ses capacités cognitives à 85 ans, l'autre développe Alzheimer à 65.
Cette variabilité, c'est exactement ce que les modèles de deep learning peuvent essayer de capturer.

En data, on dit souvent que la santé ressemble à une descente de gradient :
elle évolue dans tous les cas, mais la direction et la vitesse dépendent de plein de facteurs.
C'est cette idée qui guide notre projet.

### 1.2 Ce qu'est Alzheimer 

La maladie d'Alzheimer est la principale cause de démence dans le monde.
Elle touche **55 millions de personnes** aujourd'hui, et ce chiffre devrait tripler d'ici 2050.

Ce qu'elle fait concrètement : le cerveau rétrécit progressivement (on appelle ça l'atrophie),
certaines zones se dégradent avant d'autres, et les connexions entre neurones se cassent.
Tout ça est visible sur des IRM cérébrales  c'est ce qu'on va essayer d'apprendre à classifier.

### 1.3 Notre classification : 4 stades de sévérité

Le médecin utilise le score **CDR (Clinical Dementia Rating)** pour évaluer le niveau de démence.
On traduit ça en 4 classes dans notre dataset :

| Classe | CDR | Ce que ça veut dire |
|:---|:---|:---|
| **NonDemented** | 0 | Cerveau sain, aucun signe de démence |
| **VeryMildDemented** | 0.5 | Tout début difficile à détecter, crucial à ne pas rater |
| **MildDemented** | 1 | Démence légère, déjà bien installée |
| **ModerateDemented** | 2 | Démence modérée, stade avancé |

> ⚠️ **Point clé pour comprendre nos résultats :**
> En médecine, toutes les erreurs ne se valent pas.
> Rater un cas VeryMild (dire "pas de démence" alors qu'il y en a une),
> c'est laisser passer quelqu'un sans prise en charge à un moment où on pourrait encore agir.
> C'est pour ça qu'on regardera le **recall** autant que l'accuracy.

### 1.4 Pourquoi ce projet nous parle

On n'est pas des médecins. Mais on est des data scientists.
Et aujourd'hui, avec les bons outils, on peut aider à détecter des maladies plus tôt,
monitorer leur évolution, et personnaliser les diagnostics  parce que chaque personne vieillit différemment.
C'est cette vision qui donne du sens à ce qu'on fait ici.


---
## 2. Imports & Configuration

On importe ici toutes les bibliothèques dont on va avoir besoin.
Rien de particulier à expliquer ici  c'est du setup classique.

**Petit lexique rapide** pour la suite du notebook :
- **Accuracy** : % de bonnes prédictions toutes classes confondues
- **F1-macro** : moyenne du F1 sur chaque classe, avec le même poids pour toutes  c'est notre métrique principale car le dataset est déséquilibré
- **Recall** : sur tous les vrais cas d'une classe, combien le modèle en retrouve ?
- **Precision** : quand le modèle prédit une classe, à quelle fréquence il a raison ?
- **Overfitting** : le modèle apprend trop les exemples d'entraînement et galère sur de nouvelles images
- **Transfer Learning** : réutiliser un modèle déjà entraîné sur des millions d'images, puis l'adapter à notre problème
- **Grad-CAM** : outil qui montre visuellement où le modèle regarde dans l'image pour décider
- **GAN** : réseau qui génère de nouvelles images synthétiques pour enrichir les classes rares


In [ ]:
# ============================================================
# Installation des packages nécessaires (décommenter si besoin)
# ============================================================
# !pip install tensorflow keras scikit-learn matplotlib seaborn opencv-python-headless grad-cam

import os
import random
import warnings
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from collections import Counter

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.utils import to_categorical

# Métriques
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.utils.class_weight import compute_class_weight

# Reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Config
warnings.filterwarnings('ignore')
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

**TensorFlow chargé en version 2.15.0.**
Pas de GPU détecté l'entraînement tourne sur CPU. C'est plus lent mais ça marche.
Pour un projet avec 6400 images, c'est faisable, juste un peu long par epoch.


---
## 3. Présentation du Dataset & Justification du choix

### Source

**Alzheimer MRI Dataset**  disponible sur Kaggle.
Données dérivées des initiatives **ADNI** et **OASIS**, deux des plus grandes bases de données d'IRM cérébrales au monde.

### Pourquoi ce dataset ?

- Les images sont déjà préparées en coupes 2D → on peut se concentrer sur les modèles, pas le preprocessing 3D
- Les 4 classes correspondent exactement aux stades CDR cliniques → défendable médicalement
- Le déséquilibre entre classes est réaliste (cas modérés rares en pratique) → challenge intéressant pour nous en tant que data scientists

### Le défi principal identifié en amont

Le dataset est **très déséquilibré** :
- NonDemented >> VeryMild > Mild >> **Moderate (seulement 1%)**

Notre stratégie pour gérer ça :
1. **Class weights** : on donne plus d'importance aux classes rares pendant l'entraînement
2. **F1-macro** : notre métrique principale, qui donne le même poids à chaque classe
3. **DCGAN** en section 11 :  on génère des images synthétiques pour la classe la plus rare


In [ ]:
# ============================================================
# 3.1 — Configuration du dataset
# ============================================================
DATA_DIR = os.environ.get("MRI_DATA_DIR", "data/OriginalDataset")
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError("Set MRI_DATA_DIR to the licensed dataset with four class folders.")
CLASSES = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
NUM_CLASSES = len(CLASSES)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50

print(f"Dataset directory: {DATA_DIR}")
print(f"Classes: {CLASSES}")
print(f"Nombre de classes: {NUM_CLASSES}")

---
## 4. Prétraitement des données

### Ce qu'on fait et pourquoi

| Étape | Choix | Pourquoi |
|:---|:---|:---|
| **Chargement** | PIL, resize 224×224, niveaux de gris | 224×224 = taille standard pour ResNet50 et autres modèles pré-entraînés |
| **Normalisation** | Division par 255 → [0, 1] | Les réseaux de neurones apprennent mieux avec des valeurs entre 0 et 1 |
| **Split** | 70% train / 15% val / 15% test, stratifié | Le "stratifié" garantit que la proportion de chaque classe est respectée dans chaque split |
| **Augmentation** | Rotation ±15°, shift ±10%, zoom ±10%, flip horizontal | Simule les variations naturelles d'acquisition IRM |

> ⚠️ On n'applique **pas** de flip vertical : dans une IRM cérébrale, le haut et le bas ont une signification anatomique différente. Inverser le cerveau à l'envers n'aurait pas de sens cliniquement.

### 4.1 Chargement des images


In [ ]:
# ============================================================
# 4.1 — Chargement des images (version rapide avec PIL)
# ============================================================
from PIL import Image

def load_dataset(data_dir, classes, img_size=(224, 224)):
    images = []
    labels = []
    
    for idx, cls in enumerate(classes):
        cls_dir = os.path.join(data_dir, cls)
        if not os.path.exists(cls_dir):
            print(f"⚠️  Dossier introuvable : {cls_dir}")
            continue
        
        files = sorted(glob(os.path.join(cls_dir, "*.jpg")) +
                       glob(os.path.join(cls_dir, "*.png")) +
                       glob(os.path.join(cls_dir, "*.jpeg")))
        
        for f in files:
            img = Image.open(f).convert('L')  # Grayscale
            img = img.resize(img_size)
            images.append(np.array(img, dtype=np.float32))
            labels.append(idx)
        
        print(f"  ✅ {cls}: {len([l for l in labels if l == idx])} images chargées")
    
    images = np.array(images)
    images = np.expand_dims(images, axis=-1)  # (N, H, W, 1)
    images = images / 255.0
    labels = np.array(labels)
    
    return images, labels

# Charger le dataset
X, y = load_dataset(DATA_DIR, CLASSES, img_size=IMG_SIZE)
print(f"\nDataset chargé : {X.shape[0]} images")
print(f"Shape des images : {X.shape[1:]}")
print(f"\nDistribution des classes :")
for idx, cls in enumerate(CLASSES):
    count = np.sum(y == idx)
    print(f"  {cls}: {count} images ({count/len(y)*100:.1f}%)")


 **6400 images chargées avec succès.**

Ce que ces chiffres nous disent concrètement :

| Classe | Nb images | % | Ce que ça implique |
|:---|:---|:---|:---|
| NonDemented | 3200 | 50% | La classe dominante  le modèle va être tenté de tout prédire ici |
| VeryMildDemented | 2240 | 35% | Assez représentée mais visuellement proche de NonDemented |
| MildDemented | 896 | 14% | Minoritaire mais gérable |
| **ModerateDemented** | **64** | **1%** | **Problème critique  50× moins que NonDemented** |

**Ce qu'il faut retenir :** un modèle bête qui prédirait "NonDemented" pour toutes les images
aurait déjà **50% d'accuracy** sans avoir rien appris. C'est pour ça qu'on ne peut pas se fier
uniquement à l'accuracy dans ce projet.


In [ ]:
# Sauvegarder pour ne pas recharger à chaque fois
from pathlib import Path
CACHE_DIR = Path(os.environ.get('MRI_CACHE_DIR', 'artifacts/cache'))
CACHE_DIR.mkdir(parents=True, exist_ok=True)
np.save(CACHE_DIR / 'X.npy', X)
np.save(CACHE_DIR / 'y.npy', y)
print("Dataset sauvegardé ✅")

# Pour recharger la prochaine fois, remplace le load_dataset par :
# Cache is available under MRI_CACHE_DIR.
# Cache is available under MRI_CACHE_DIR.

### 4.2 Split Train / Validation / Test

On découpe le dataset en 3 parties :
- **Train (70%)** : les images sur lesquelles le modèle apprend
- **Validation (15%)** : les images qu'on regarde pendant l'entraînement pour vérifier qu'il ne mémorise pas
- **Test (15%)** : les images qu'on ne touche pas jusqu'à l'évaluation finale  c'est le vrai score

Le mot clé ici est **stratified** : on s'assure que chaque classe garde la même proportion dans les 3 splits.
Sans ça, la classe ModerateDemented (64 images au total) pourrait avoir 0 exemples dans le test.


In [ ]:
# ============================================================
# 4.2 — Stratified Split : 70% train, 15% val, 15% test
# ============================================================
# Premier split : 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

# Second split : 50/50 du temp → 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

# One-hot encoding
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat = to_categorical(y_val, NUM_CLASSES)
y_test_cat = to_categorical(y_test, NUM_CLASSES)

print(f"Train : {X_train.shape[0]} images")
print(f"Val   : {X_val.shape[0]} images")
print(f"Test  : {X_test.shape[0]} images")

# Vérification du stratify
for split_name, split_y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    dist = Counter(split_y)
    print(f"\n{split_name} distribution :")
    for idx in sorted(dist.keys()):
        print(f"  {CLASSES[idx]}: {dist[idx]}")

 **Split effectué :**
- Train : 4480 images (dont seulement **45 ModerateDemented**)
- Val : 960 images
- Test : 960 images

**Attention sur ModerateDemented :**
Au total il y avait 64 images. Après split :
- Train : 45 / Val : 10 / Test : 9

C'est très peu pour apprendre. Le GAN qu'on utilisera plus tard (section 11)
sera entraîné sur ces **45 images** seulement  ce qui expliquera ses limites.


### 4.3 Data Augmentation

La data augmentation crée des variations artificielles des images d'entraînement.
**Elle s'applique uniquement sur le train**  jamais sur val ni test, sinon on tricherait.

Chaque transformation a une raison concrète :
- **Rotation ±15°** : dans la vraie vie, les patients ne sont pas positionnés exactement pareil dans le scanner
- **Shift ±10%** : le cerveau peut être légèrement décalé dans le champ de vue
- **Zoom ±10%** : la distance entre le scanner et le patient varie
- **Flip horizontal** : le cerveau est approximativement symétrique gauche/droite

L'idée : faire croire au modèle qu'il a vu plus d'images différentes qu'il n'en a vraiment vues.
Ça l'aide à généraliser au lieu de mémoriser.


In [ ]:
# ============================================================
# 4.3 — Data Augmentation
# ============================================================
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator()  # Pas d'augmentation sur val/test

train_generator = train_datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE, seed=SEED)
val_generator = val_datagen.flow(X_val, y_val_cat, batch_size=BATCH_SIZE, seed=SEED)

# Visualiser des exemples augmentés
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Exemples de Data Augmentation", fontsize=14, fontweight='bold')

sample_img = X_train[0:1]
for i, ax in enumerate(axes.flat):
    augmented = train_datagen.random_transform(sample_img[0])
    ax.imshow(augmented.squeeze(), cmap='gray')
    ax.axis('off')
    ax.set_title(f"Aug. {i+1}", fontsize=9)

plt.tight_layout()
plt.show()

 **Data augmentation configurée.**

Les images ci-dessus montrent le même cerveau avec différentes transformations appliquées.
On voit clairement les rotations légères, les petits décalages et les variations de zoom.

**Ce qui est important à noter :** le contenu médical reste identique.
On ne crée pas un nouveau cerveau  on crée de nouvelles vues du même cerveau.
C'est acceptable car en pratique, un IRM d'un même patient pris deux jours différents
aurait ces mêmes petites variations.


---
## 5. Analyse Exploratoire (EDA)

Avant de construire quoi que ce soit, on regarde les données.
C'est une règle de base en data science : **comprendre avant de modéliser**.

Ce qu'on cherche ici :
- Est-ce que les classes sont équilibrées ? (on sait déjà que non)
- Est-ce qu'on voit une différence visuelle entre les stades ?
- Les intensités de pixels sont-elles différentes selon la classe ?

Les réponses à ces questions vont directement influencer nos choix de modélisation.


In [ ]:
# ============================================================
# 5.1 — Distribution des classes
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts = [np.sum(y == i) for i in range(NUM_CLASSES)]
colors = ['#e67e22', '#e74c3c', '#2ecc71', '#f39c12']

bars = axes[0].bar(CLASSES, class_counts, color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title("Distribution des classes", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Nombre d'images")
axes[0].tick_params(axis='x', rotation=20)
for bar, count in zip(bars, class_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
                 str(count), ha='center', fontweight='bold')

axes[1].pie(class_counts, labels=CLASSES, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 10})
axes[1].set_title("Répartition proportionnelle", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

ratio = max(class_counts) / min(class_counts)
print(f"\nRatio de déséquilibre max/min : {ratio:.1f}x")
print("⚠️  Déséquilibre MASSIF détecté — class weighting + data augmentation + GAN indispensables.")

**Ce que le graphique nous dit :**

Le déséquilibre est massif. **Ratio max/min = 50:1** (3200 NonDemented vs 64 ModerateDemented).

**Conséquence directe pour nos modèles :**
Un modèle qui répondrait "NonDemented" à toutes les images aurait **50% d'accuracy**.
C'est exactement ce que fait notre baseline (spoiler de la section 6).
C'est pour ça qu'on utilise le **F1-macro** comme métrique principale  il donne le même poids
à toutes les classes, même les plus rares.

**Les 3 actions qu'on prend pour compenser :**
1. Class weights pendant l'entraînement (la classe rare compte plus dans le calcul de la loss)
2. F1-macro comme critère de comparaison
3. GAN pour générer des images synthétiques ModerateDemented 


In [ ]:
# ============================================================
# 5.2 — Visualisation d'exemples par classe
# ============================================================
fig, axes = plt.subplots(4, 5, figsize=(16, 13))
fig.suptitle("Exemples d'IRM par classe", fontsize=15, fontweight='bold')

for row, (cls_idx, cls_name) in enumerate(enumerate(CLASSES)):
    cls_images = X[y == cls_idx]
    n_available = min(5, len(cls_images))
    sample_indices = np.random.choice(len(cls_images), size=n_available, replace=False)
    
    for col in range(5):
        if col < n_available:
            axes[row, col].imshow(cls_images[sample_indices[col]].squeeze(), cmap='gray')
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(cls_name, fontsize=11, fontweight='bold',
                                       rotation=0, labelpad=100)

plt.tight_layout()
plt.show()

 **Ce qu'on voit sur les IRM :**

En regardant les 4 lignes (une par classe), la première chose qui frappe :
**les cerveaux se ressemblent beaucoup à l'œil nu.**

Les différences subtiles qu'on peut noter :
- Les ventricules (les zones sombres en forme de papillon au centre) semblent légèrement plus grands dans les stades avancés
- L'espace entre le cerveau et le crâne paraît parfois un peu plus large pour les stades modérés
- Entre **NonDemented** et **VeryMildDemented**, c'est pratiquement impossible à distinguer à l'œil

**Ce que ça nous dit pour la suite :**
Si même un humain a du mal à voir la différence, le modèle va aussi avoir du mal.
C'est précisément pour ça qu'on utilise du deep learning : il peut détecter des patterns
microscopiques qu'on ne voit pas directement.


**Conclusion de l'observation visuelle :**

La frontière entre **NonDemented** et **VeryMildDemented** est la plus floue visuellement.
C'est la confusion qu'on retrouvera systématiquement dans toutes nos matrices de confusion.
Ce n'est pas un bug du modèle c'est un problème inhérent au dataset.

Même les médecins ont besoin d'un bilan cognitif complet (pas juste une IRM)
pour poser un diagnostic fiable à ce stade précoce.


In [ ]:
# ============================================================
# 5.3 — Analyse des intensités de pixels par classe
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("Distribution des intensités moyennes par classe", fontsize=13, fontweight='bold')

for idx, cls_name in enumerate(CLASSES):
    cls_images = X[y == idx]
    mean_intensities = cls_images.mean(axis=(1, 2, 3))
    axes[idx].hist(mean_intensities, bins=30, color=colors[idx], alpha=0.7, edgecolor='black')
    axes[idx].set_title(f"{cls_name}\n(n={len(cls_images)})", fontsize=10)
    axes[idx].set_xlabel("Intensité moyenne")
    axes[idx].axvline(mean_intensities.mean(), color='red', linestyle='--',
                      label=f"μ={mean_intensities.mean():.3f}")
    axes[idx].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\n📊 Analyse des difficultés potentielles :")
print("- NonDemented et VeryMildDemented sont visuellement très proches")
print("- ModerateDemented n'a que 64 images → risque d'overfitting sur cette classe")
print("- Le modèle devra apprendre des différences subtiles d'atrophie corticale")

### Synthèse de l'exploration 

**Problème 1 : déséquilibre des classes**
3200 NonDemented vs 64 ModerateDemented = rapport 50:1.
→ On utilisera class weights, F1-macro, et GAN pour compenser.

**Problème 2 : classes visuellement proches**
NonDemented et VeryMildDemented se ressemblent beaucoup sur une coupe 2D.
→ On s'attend à des confusions fréquentes entre ces deux classes dans toutes nos matrices.

**Problème 3 : pas assez de données pour ModerateDemented**
64 images dans tout le dataset, 45 dans le train.
→ Le GAN sera limité par cette quantité très faible.

Ces 3 problèmes expliquent à l'avance pourquoi aucun de nos modèles ne sera parfait.
C'est normal. L'objectif est de progresser à chaque étape et de comprendre pourquoi.


**Analyse des intensités de pixels :**

Les 4 histogrammes montrent que les intensités moyennes sont très proches entre les classes :
- MildDemented : **μ = 0.265**
- ModerateDemented : **μ = 0.273**
- VeryMildDemented : **μ = 0.275**
- NonDemented : **μ = 0.291**

L'écart total entre la plus faible et la plus haute : **seulement 0.026** sur une échelle de 0 à 1.

**Ce que ça veut dire concrètement :**
Le modèle ne peut pas juste regarder "si l'image est claire ou sombre" pour décider.
Il doit apprendre des **patterns structurels locaux**  des textures, des contours,
des formes spécifiques à certaines zones du cerveau.
C'est exactement ce que les couches convolutives d'un CNN sont capables de faire.

**Note sur ModerateDemented :** sa courbe est irrégulière car elle est basée sur seulement 64 images.
Avec si peu de données, la distribution est instable  un résultat de plus en faveur du GAN.


---
## 6. Modèle Baseline :  CNN Simple

### Pourquoi on commence par un modèle simple ?

En data science, on ne commence jamais directement avec le modèle le plus complexe.
On part toujours d'un **point de référence simple** pour savoir ce qu'on gagne ensuite.

Si on commençait avec ResNet50, on ne saurait pas si c'est vraiment mieux qu'un CNN basique.
Le baseline permet de mesurer concrètement l'apport de chaque amélioration.

### Architecture du baseline

```
Conv2D(32 filtres, 3×3) + ReLU + MaxPool(2×2)
Conv2D(64 filtres, 3×3) + ReLU + MaxPool(2×2)
Flatten → Dense(128) + ReLU → Dropout(0.5) → Dense(4, softmax)
```

**Ce qui est absent volontairement :** BatchNorm, data augmentation, GlobalAvgPool.
Ce sont exactement les améliorations qu'on ajoutera au CNN amélioré (section 7).

**23,9 millions de paramètres**  c'est énorme pour un si petit réseau.
Pourquoi ? À cause du `Flatten` → `Dense(128)` qui crée une couche dense de 186 624 × 128 = **23,9M connexions**.
C'est une des choses qu'on améliorera avec le GlobalAveragePooling ensuite.


In [ ]:
# ============================================================
# 6.1 — Construction du modèle baseline
# ============================================================
def build_baseline_cnn(input_shape, num_classes):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape, name='baseline_conv1'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu', name='baseline_conv2'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu', name='baseline_dense1'),
        layers.Dense(num_classes, activation='softmax', name='baseline_output')
    ], name='baseline_cnn')
    return model

INPUT_SHAPE = (224, 224, 1)
baseline_model = build_baseline_cnn(INPUT_SHAPE, NUM_CLASSES)
baseline_model.summary()

In [ ]:
# ============================================================
# 6.2 — Compilation et entraînement
# ============================================================
baseline_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Class weights pour gérer le déséquilibre
class_weights_array = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights_array))
print(f"Class weights : {class_weights}")

baseline_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

baseline_history = baseline_model.fit(
    train_generator,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=val_generator,
    validation_steps=len(X_val) // BATCH_SIZE,
    class_weight=class_weights,
    callbacks=baseline_callbacks,
    verbose=1
)

 **Ce qu'on voit pendant l'entraînement du baseline :**

La val_loss tourne autour de **1.385** pendant presque tout l'entraînement.
Elle ne descend pas vraiment. La val_accuracy saute entre 0.35 et 0.50 sans tendance claire.

**Traduction simple :**
Le modèle apprend à survivre statistiquement dans un dataset déséquilibré,
mais il ne construit pas une vraie compréhension des 4 classes.
Il oscille autour d'une stratégie du type : *"je prédirai surtout NonDemented, c'est ma meilleure chance"*.

Les class weights l'empêchent de tomber totalement dans ce piège,
mais le réseau n'est pas assez puissant pour en sortir complètement.


In [ ]:
# ============================================================
# 6.3 — Courbes d'apprentissage
# ============================================================
def plot_training_history(history, model_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Courbes d'apprentissage — {model_name}", fontsize=14, fontweight='bold')
    
    ax1.plot(history.history['loss'], label='Train Loss', linewidth=2)
    ax1.plot(history.history['val_loss'], label='Val Loss', linewidth=2, linestyle='--')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss')
    ax1.legend(); ax1.grid(True, alpha=0.3)
    
    ax2.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    ax2.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, linestyle='--')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Accuracy')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_training_history(baseline_history, "Baseline CNN")

 **Lecture des courbes d'apprentissage du baseline :**

**Graphique Loss :** la val_loss (en pointillés) est quasi-plate dès le début.
Elle ne descend pas de façon consistante. Le modèle ne progresse pas vraiment.

**Graphique Accuracy :** la val_accuracy oscille entre 0.35 et 0.50 sans jamais construire de tendance nette.
C'est le signe typique d'un modèle qui ne converge pas il cherche une solution mais ne la trouve pas.

**Ce que ça nous dit pour la suite :**
Ces courbes justifient complètement les améliorations de la section 7.
Sans BatchNorm, sans augmentation et avec trop de paramètres dans les couches denses,
le modèle est coincé. On peut faire mieux.


**Ce qu'il faut retenir de ces courbes :**

La val_accuracy autour de 0.35–0.50 peut sembler "pas si mal" au premier regard.
Mais rappelons que prédire "NonDemented" pour tout le monde donnerait déjà 50%.

Le baseline n'a donc pas vraiment appris à distinguer les stades.
Il a appris à survivre dans un dataset déséquilibré en favorisant la classe majoritaire.
C'est exactement ce qu'on va confirmer dans l'évaluation ci-dessous.


In [ ]:
# ============================================================
# 6.4 — Évaluation du baseline
# ============================================================
def evaluate_model(model, X_test, y_test, y_test_cat, classes, model_name):
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    print(f"\n{'='*60}")
    print(f"  ÉVALUATION — {model_name}")
    print(f"{'='*60}")
    
    test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
    print(f"\nTest Loss     : {test_loss:.4f}")
    print(f"Test Accuracy : {test_acc:.4f}")
    print(f"\nClassification Report :")
    print(classification_report(y_test, y_pred, target_names=classes, digits=4))
    
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f"Matrice de Confusion — {model_name}", fontsize=13, fontweight='bold')
    ax.set_xlabel("Prédiction"); ax.set_ylabel("Vérité terrain")
    plt.xticks(rotation=20); plt.yticks(rotation=0)
    plt.tight_layout(); plt.show()
    
    return {
        'model_name': model_name,
        'accuracy': test_acc, 'loss': test_loss,
        'f1_macro': f1_score(y_test, y_pred, average='macro'),
        'precision_macro': precision_score(y_test, y_pred, average='macro'),
        'recall_macro': recall_score(y_test, y_pred, average='macro'),
        'y_pred': y_pred, 'y_pred_proba': y_pred_proba
    }

baseline_results = evaluate_model(baseline_model, X_test, y_test, y_test_cat, CLASSES, "Baseline CNN")

 **Résultats du Baseline  ce que les chiffres disent vraiment :**

| Métrique | Valeur | Interprétation |
|:---|:---|:---|
| Accuracy | 49.90% | Quasi identique à prédire toujours "NonDemented" |
| F1-macro | 0.166 | Très mauvais  3 classes sur 4 sont ignorées |
| Recall MildDemented | 0.000 | Le modèle ne détecte aucun cas Mild |
| Recall ModerateDemented | 0.000 | Idem  aucun cas Moderate détecté |
| Recall NonDemented | 0.998 | Le modèle prédit presque tout en NonDemented |
| Recall VeryMildDemented | 0.000 | Aucun cas VeryMild détecté |

**Le verdict :**
Le baseline a appris une seule chose : prédire "NonDemented".
Sur 960 images de test, il en classe ~479 correctement, presque toutes parce qu'elles sont NonDemented.

Ce n'est pas un modèle de classification  c'est un modèle qui dit toujours la même chose.
C'est pour ça qu'on a besoin du CNN amélioré.


**La matrice de confusion le confirme visuellement :**

Toute la diagonale est vide sauf la case NonDemented.
Les 135 cas MildDemented, les 9 cas ModerateDemented et les 336 VeryMildDemented
sont tous classés ailleurs  principalement en NonDemented.

**Point important pour la discussion :**
Le baseline n'est pas "raté"  il joue exactement son rôle.
Il nous montre où on en est sans amélioration, et il justifie chaque choix qu'on fait ensuite.


---
## 7. CNN Amélioré

Le baseline nous a montré ses limites. On sait maintenant ce qu'il faut corriger :
- Trop de paramètres (23.9M → on va réduire avec GlobalAveragePooling)
- Pas de BatchNorm (l'apprentissage n'est pas stable)
- Pas de régularisation profonde (overfitting)
- Architecture trop simple (2 blocs seulement)

On va corriger tout ça, **une amélioration à la fois**, pour que le gain soit mesurable.


In [ ]:
# ============================================================
# 7.1 — Construction du CNN amélioré
# ============================================================
from tensorflow.keras import regularizers

def build_improved_cnn(input_shape, num_classes):
    model = models.Sequential(name='improved_cnn')
    
    # Bloc 1
    model.add(layers.Conv2D(32, (3,3), padding='same', input_shape=input_shape,
                            kernel_regularizer=regularizers.l2(1e-4), name='imp_conv1_1'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(32, (3,3), padding='same',
                            kernel_regularizer=regularizers.l2(1e-4), name='imp_conv1_2'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))
    
    # Bloc 2
    model.add(layers.Conv2D(64, (3,3), padding='same',
                            kernel_regularizer=regularizers.l2(1e-4), name='imp_conv2_1'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(64, (3,3), padding='same',
                            kernel_regularizer=regularizers.l2(1e-4), name='imp_conv2_2'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.3))
    
    # Bloc 3
    model.add(layers.Conv2D(128, (3,3), padding='same',
                            kernel_regularizer=regularizers.l2(1e-4), name='imp_conv3_1'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(128, (3,3), padding='same',
                            kernel_regularizer=regularizers.l2(1e-4), name='imp_conv3_2'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.35))
    
    # Bloc 4
    model.add(layers.Conv2D(256, (3,3), padding='same',
                            kernel_regularizer=regularizers.l2(1e-4), name='imp_conv4_1'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.4))
    
    # Classification
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4)))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(num_classes, activation='softmax', name='imp_output'))
    
    return model

improved_model = build_improved_cnn(INPUT_SHAPE, NUM_CLASSES)
improved_model.summary()
print(f"\nParamètres — Baseline : {baseline_model.count_params():,}")
print(f"Paramètres — Amélioré : {improved_model.count_params():,}")

### Architecture du CNN amélioré  explication de chaque choix

| Amélioration | Ce qu'elle fait concrètement |
|:---|:---|
| **4 blocs conv** (vs 2) | Le modèle peut apprendre des features plus complexes et abstraites |
| **Batch Normalization** | Normalise les valeurs entre chaque couche → entraînement plus stable et plus rapide |
| **Dropout progressif** (0.25 → 0.5) | Désactive aléatoirement des neurones → force le modèle à ne pas mémoriser |
| **GlobalAveragePooling** | Remplace Flatten → passe de 23.9M à 651K paramètres (-97%) |
| **L2 Regularization** | Pénalise les poids trop grands → meilleure généralisation |
| **ReduceLROnPlateau** | Réduit le taux d'apprentissage quand il stagne → convergence plus fine |

**Le point le plus important : GlobalAveragePooling**
Le baseline avait 23.9 millions de paramètres dans ses couches denses.
Le CNN amélioré en a **651 000**. Avec moins de paramètres, il y a moins de risque d'overfitting.
C'est contre-intuitif mais souvent vrai : **moins c'est plus**.


In [ ]:
# ============================================================
# 7.2 — Entraînement du CNN amélioré
# ============================================================
improved_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

improved_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

improved_history = improved_model.fit(
    train_generator,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=val_generator,
    validation_steps=len(X_val) // BATCH_SIZE,
    class_weight=class_weights,
    callbacks=improved_callbacks,
    verbose=1
)

plot_training_history(improved_history, "CNN Amélioré")

 **Le CNN amélioré est en train de s'entraîner...**

Ce qu'on attend de voir dans les courbes par rapport au baseline :
- La val_loss doit descendre plus régulièrement (pas rester plate)
- L'écart entre train et val ne doit pas être trop grand (sinon overfitting)
- Les métriques par classe doivent montrer que le modèle détecte autre chose que NonDemented

Si ces 3 conditions sont remplies, les améliorations ont fonctionné.


In [ ]:
# ============================================================
# 7.3 — Évaluation du CNN amélioré
# ============================================================
improved_results = evaluate_model(improved_model, X_test, y_test, y_test_cat, CLASSES, "CNN Amélioré")

 **Résultats du CNN Amélioré  la progression est nette :**

| Classe | Recall Baseline | Recall CNN amélioré | Progression |
|:---|:---|:---|:---|
| MildDemented | 0.000 | **0.933** | +93%  126/135 cas détectés |
| ModerateDemented | 0.000 | **0.778** | +78%  7/9 cas détectés |
| NonDemented | 0.998 | 0.579 | Légère baisse  normal, le modèle est moins biaisé |
| VeryMildDemented | 0.000 | **0.262** | +26%  encore faible mais le modèle commence à voir |

**F1-macro : 0.166 → 0.572**  le gain principal, c'est là qu'il est.

**Ce que ça signifie concrètement :**
Le modèle ne prédit plus seulement "NonDemented". Il commence à différencier les stades.
Mild et Moderate sont bien récupérés. VeryMild reste difficile  on y reviendra.

**La faiblesse qui reste :**
VeryMildDemented avec recall 0.26 : 248 cas sur 336 sont mal classés (en NonDemented ou Mild).
Ce n'est pas une surprise. C'est la frontière la plus floue visuellement dans tout le dataset.


**Pourquoi le F1-macro est notre vraie métrique de succès ici :**

L'accuracy du CNN amélioré est de 51.98% presque identique au baseline (49.90%).
Pourtant ce sont deux modèles complètement différents.

Le baseline avait 50% en prédisant tout en NonDemented.
Le CNN amélioré a 52% en reconnaissant réellement les 4 classes.

Cette différence d'accuracy de 2% cache une différence de F1-macro de +0.41.
**C'est pour ça qu'on regarde le F1-macro.**


**Lecture de la matrice de confusion du CNN amélioré :**

Par rapport au baseline où toute la diagonale était vide sauf NonDemented,
on voit maintenant **plusieurs classes correctement détectées** :
- 126 MildDemented bien classés (sur 135)
- 7 ModerateDemented bien classés (sur 9)
- 278 NonDemented bien classés (sur 480)
- 88 VeryMildDemented bien classés (sur 336)

**La confusion principale :** 146 VeryMild sont classés comme Mild, 101 comme NonDemented.
C'est la frontière la plus difficile de tout le projet  attendu d'après l'EDA.

**Ce modèle est notre meilleur en F1-macro.** On continue pour voir ce que ResNet50 apporte.


---
## 8. Transfer Learning : ResNet50

### L'idée derrière le Transfer Learning

ResNet50 a été entraîné sur **1.2 million d'images** d'ImageNet (chats, voitures, plantes...).
Il a appris à reconnaître des textures, des bords, des formes, des structures. ( 50 car il a 50 couches de neurones)

On réutilise tout ce savoir, et on l'adapte à nos IRM cérébrales.
C'est comme apprendre à conduire une voiture automatique quand on sait déjà conduire en manuelle :
on repart pas de zéro, on adapte.

### Pourquoi ResNet50 et pas VGG16 ou VGG19 ?

| Critère | ResNet50 | VGG16 |
|:---|:---|:---|
| Paramètres | ~25M | ~138M |
| Connexions résiduelles | ✅ évite le vanishing gradient | ❌ |
| Adapté à nos données limitées | ✅ moins d'overfitting | ✗ trop lourd |

ResNet50 est plus compact et plus robuste. Meilleur choix pour notre contexte.

### Stratégie en 2 phases

- **Phase 1** : on gèle toute la base ResNet, on entraîne seulement la tête de classification
- **Phase 2** : on dégèle les 30 dernières couches pour un fine-tuning fin

Pourquoi 2 phases ? Si on fine-tune tout dès le début avec un LR normal,
on risque de "casser" les représentations qu'il a déjà apprises sur ImageNet.


In [ ]:
# ============================================================
# 8.1 — Préparer les données pour ResNet50 (Grayscale → RGB)
# ============================================================
X_train_rgb = np.repeat(X_train, 3, axis=-1)
X_val_rgb = np.repeat(X_val, 3, axis=-1)
X_test_rgb = np.repeat(X_test, 3, axis=-1)

print(f"Shape après conversion RGB :")
print(f"  Train : {X_train_rgb.shape}")
print(f"  Val   : {X_val_rgb.shape}")
print(f"  Test  : {X_test_rgb.shape}")

train_datagen_rgb = ImageDataGenerator(
    rotation_range=15, width_shift_range=0.1,
    height_shift_range=0.1, zoom_range=0.1,
    horizontal_flip=True, fill_mode='nearest'
)
train_gen_rgb = train_datagen_rgb.flow(X_train_rgb, y_train_cat, batch_size=BATCH_SIZE, seed=SEED)
val_gen_rgb = ImageDataGenerator().flow(X_val_rgb, y_val_cat, batch_size=BATCH_SIZE, seed=SEED)

### Pourquoi convertir en RGB ?

ResNet50 attend **3 canaux** en entrée (R, G, B  comme une photo couleur).
Nos IRM sont en **niveaux de gris** (1 seul canal).

La solution : on duplique le même canal 3 fois.
Le modèle "croit" recevoir une image couleur, mais les 3 canaux sont identiques.

**Ce que ça change :** rien au contenu médical. Le cerveau est exactement le même.
On adapte juste le format pour que ResNet50 accepte nos données.


In [ ]:
# ============================================================
# 8.2 — Construction du modèle ResNet50
# ============================================================
def build_resnet50_transfer(input_shape, num_classes):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    
    # Phase 1 : geler toutes les couches
    for layer in base_model.layers:
        layer.trainable = False
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ], name='resnet50_transfer')
    
    return model, base_model

INPUT_SHAPE_RGB = (224, 224, 3)
resnet_model, resnet_base = build_resnet50_transfer(INPUT_SHAPE_RGB, NUM_CLASSES)
resnet_model.summary()

trainable = sum([tf.keras.backend.count_params(w) for w in resnet_model.trainable_weights])
non_trainable = sum([tf.keras.backend.count_params(w) for w in resnet_model.non_trainable_weights])
print(f"\nParamètres entraînables : {trainable:,}")
print(f"Paramètres gelés       : {non_trainable:,}")

 **ResNet50 chargé avec les poids ImageNet.**

Ce qu'on voit dans le résumé du modèle :
- **23.6M paramètres gelés** (la base ResNet on ne les touche pas en phase 1)
- **~1.2M paramètres entraînables** (notre tête de classification)

**Phase 1 : on entraîne seulement la petite tête.**
C'est comme brancher un GPS de voiture sur une moto : on adapte l'interface,
pas le moteur. Le moteur (ResNet) reste intact.


In [ ]:
# ============================================================
# 8.3 — Phase 1 : Entraînement du top (base gelée)
# ============================================================
print("=" * 60)
print("  PHASE 1 : Entraînement du top uniquement")
print("=" * 60)

resnet_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

resnet_history_p1 = resnet_model.fit(
    train_gen_rgb,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=30,
    validation_data=val_gen_rgb,
    validation_steps=len(X_val) // BATCH_SIZE,
    class_weight=class_weights,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
    ],
    verbose=1
)

plot_training_history(resnet_history_p1, "ResNet50 — Phase 1 (top only)")

 **Phase 1  Base gelée, seule la tête s'entraîne :**

On teste ici si les features apprises sur des millions de photos naturelles
peuvent servir de point de départ pour des IRM cérébrales.

Ce n'est pas évident. Une IRM c'est en niveaux de gris, avec des structures très différentes
d'un chat ou d'une voiture. Mais les couches profondes de ResNet apprennent des features
génériques (bords, textures, formes) qui peuvent être partiellement réutilisables.

La phase 2 permettra au réseau de s'adapter plus finement à nos IRM.


In [ ]:
# ============================================================
# 8.4 — Phase 2 : Fine-tuning (dégel des 30 dernières couches)
# ============================================================
print("=" * 60)
print("  PHASE 2 : Fine-tuning")
print("=" * 60)

for layer in resnet_base.layers[-30:]:
    layer.trainable = True

resnet_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

resnet_history_p2 = resnet_model.fit(
    train_gen_rgb,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=30,
    validation_data=val_gen_rgb,
    validation_steps=len(X_val) // BATCH_SIZE,
    class_weight=class_weights,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)
    ],
    verbose=1
)

plot_training_history(resnet_history_p2, "ResNet50 — Phase 2 (fine-tuning)")

 **Phase 2  Fine-tuning des 30 dernières couches :**

On déverrouille une partie de ResNet pour qu'il s'adapte mieux à nos IRM.

**Pourquoi seulement les 30 dernières couches ?**
Les premières couches apprennent des features très génériques (bords, contrastes).
Ces features sont utiles pour n'importe quelle image  IRM ou photo de chat.
Les dernières couches apprennent des features plus spécifiques au domaine.
C'est là qu'il faut adapter.

**Pourquoi un LR très faible (1e-5) ?**
Si on utilise un LR normal, on risque de détruire ce que ResNet a appris.
Le LR très faible permet des ajustements fins sans tout casser.


In [ ]:
# ============================================================
# 8.5 — Évaluation ResNet50
# ============================================================
resnet_results = evaluate_model(resnet_model, X_test_rgb, y_test, y_test_cat, CLASSES, "ResNet50 Transfer Learning")

**Résultats ResNet50  Une surprise intéressante :**

| Métrique | CNN Amélioré | ResNet50 |
|:---|:---|:---|
| Accuracy | 51.98% | **54.79%** ✅ |
| F1-macro | **0.572** ✅ | 0.447 |
| Recall ModerateDemented | 0.778 | **1.000** ✅ |
| Recall VeryMildDemented | 0.262 | 0.327 |

**ResNet50 a la meilleure accuracy, mais le CNN amélioré a le meilleur F1-macro.**

**Pourquoi ResNet50 détecte 100% des cas ModerateDemented ?**
Recall 1.0 sur Moderate mais precision 0.17 : il prédit "Moderate" très souvent,
donc il les retrouve tous mais il génère beaucoup de faux positifs.
C'est une stratégie conservatrice : mieux vaut sur-diagnostiquer que rater.

**Verdict :**
Pour notre objectif (équilibre entre toutes les classes), le CNN amélioré reste préférable.
ResNet50 est intéressant si l'objectif est de ne manquer aucun cas grave.


---
## 9. Évaluation Comparative

On a maintenant 3 modèles entraînés. Il est temps de les comparer sérieusement.

**Notre critère principal : F1-macro**
Dans un dataset déséquilibré comme le nôtre, l'accuracy peut être trompeuse.
Le F1-macro donne le même poids à chaque classe  c'est ce qui nous intéresse vraiment.

**Ce qu'on cherche :**
Quel modèle est le plus utile pour notre problème réel ?
Pas forcément le plus précis en global, mais le plus équilibré sur toutes les classes.


In [ ]:
# ============================================================
# 9.1 — Tableau comparatif
# ============================================================
all_results = [baseline_results, improved_results, resnet_results]

comparison_df = pd.DataFrame([{
    'Modèle': r['model_name'],
    'Accuracy': f"{r['accuracy']:.4f}",
    'F1 (macro)': f"{r['f1_macro']:.4f}",
    'Precision (macro)': f"{r['precision_macro']:.4f}",
    'Recall (macro)': f"{r['recall_macro']:.4f}",
    'Loss': f"{r['loss']:.4f}"
} for r in all_results])

print("=" * 80)
print("  COMPARAISON DES MODÈLES")
print("=" * 80)
print(comparison_df.to_string(index=False))

 **Lecture du tableau comparatif :**

| Modèle | Accuracy | F1-macro | Ce qu'il faut retenir |
|:---|:---|:---|:---|
| Baseline | 49.90% | 0.166 | Prédit quasi-tout en NonDemented |
| CNN Amélioré | 51.98% | **0.572** | Meilleur équilibre entre les 4 classes |
| ResNet50 | **54.79%** | 0.447 | Meilleure accuracy globale mais moins équilibré |

**Le paradoxe apparent :**
ResNet50 a une meilleure accuracy que le CNN amélioré.
Mais le CNN amélioré a un meilleur F1-macro.

Ce n'est pas une contradiction c'est précisément pourquoi on utilise plusieurs métriques.
L'accuracy regarde combien on a de bonnes réponses en total.
Le F1-macro regarde si toutes les classes sont bien traitées.

**Dans notre contexte médical : le F1-macro prime.**


In [ ]:
# ============================================================
# 9.2 — Visualisation comparative
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Comparaison des performances", fontsize=15, fontweight='bold')

model_names = [r['model_name'] for r in all_results]
colors_bar = ['#3498db', '#2ecc71', '#e74c3c']

accs = [r['accuracy'] for r in all_results]
bars = axes[0].bar(model_names, accs, color=colors_bar, edgecolor='black', alpha=0.85)
axes[0].set_title("Test Accuracy"); axes[0].set_ylim(0, 1)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f"{acc:.3f}", ha='center', fontweight='bold')

f1s = [r['f1_macro'] for r in all_results]
bars = axes[1].bar(model_names, f1s, color=colors_bar, edgecolor='black', alpha=0.85)
axes[1].set_title("F1-Score (macro)"); axes[1].set_ylim(0, 1)
for bar, f1 in zip(bars, f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f"{f1:.3f}", ha='center', fontweight='bold')

plt.tight_layout(); plt.show()

 **Ce que le barplot nous montre :**

Les deux graphiques (Accuracy et F1-macro) ne classent pas les modèles dans le même ordre.
C'est le message le plus important de cette section.

- En **accuracy** : ResNet50 > CNN amélioré > Baseline
- En **F1-macro** : CNN amélioré > ResNet50 > Baseline

Le baseline est dernier dans les deux cas  c'est cohérent.

**Retenir :** si on avait regardé seulement l'accuracy, on aurait choisi ResNet50.
En regardant le F1-macro, on choisit le CNN amélioré.
Les métriques qu'on choisit décident du "meilleur modèle".
C'est pour ça qu'on doit les choisir en fonction du problème réel.


In [ ]:
# ============================================================
# 9.3 — Matrices de confusion côte à côte
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle("Matrices de confusion comparées", fontsize=15, fontweight='bold')

for idx, r in enumerate(all_results):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[idx])
    axes[idx].set_title(r['model_name'], fontsize=11, fontweight='bold')
    axes[idx].set_xlabel("Prédiction")
    axes[idx].tick_params(axis='x', rotation=30)
    if idx == 0: axes[idx].set_ylabel("Vérité terrain")

plt.tight_layout(); plt.show()

**Lecture des 3 matrices de confusion côte à côte :**

La diagonale = ce que le modèle a bien classé.
Hors diagonale = les erreurs.

**Baseline :** toute la diagonale est vide sauf la case NonDemented.
→ Le modèle dit "NonDemented" pour tout le monde.

**CNN Amélioré :** plusieurs cases sur la diagonale s'allument.
→ MildDemented et ModerateDemented sont bien récupérés.
→ VeryMildDemented reste la classe la plus difficile.

**ResNet50 :** la diagonale est globalement meilleure.
→ Mais on voit des erreurs déplacées  VeryMild est encore confondu avec Mild et NonDemented.

**La confusion persistante :** VeryMild ↔ NonDemented dans tous les modèles.
C'est la frontière clinique la plus subtile du dataset. Tous nos modèles la ratent partiellement.


---
## 10. Interprétabilité : Grad-CAM

### Pourquoi on ne s'arrête pas aux métriques ?

Un modèle peut avoir un bon F1-macro pour de mauvaises raisons.
Exemple : si une IRM a toujours un artefact dans un coin, et que cet artefact est corrélé
avec le stade de la maladie dans notre dataset, le modèle peut apprendre à regarder cet artefact.
Il aurait de bonnes métriques, mais ne ferait pas de la vraie classification.

**Grad-CAM résout ce problème.** Il montre visuellement où le modèle regarde pour décider.

### Comment ça marche (version simple)

Grad-CAM calcule quelles régions de l'image ont eu le plus d'influence sur la décision du modèle.
Il produit une carte de chaleur qu'on superpose à l'image originale :
- 🔴 Rouge = zone très importante pour la décision
- 🔵 Bleu = zone peu importante

**On n'est pas des médecins.** On ne va pas diagnostiquer des zones du cerveau.
On vérifie juste que le modèle regarde des zones structurées du cerveau,
pas les bords noirs ou du bruit visuel.


In [ ]:
# ============================================================
# 10.1 — Implémentation Grad-CAM
# ============================================================
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def display_gradcam(img, heatmap, title="Grad-CAM", alpha=0.4, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_colored = plt.cm.jet(heatmap_resized)[:, :, :3]
    
    if img.ndim == 2 or img.shape[-1] == 1:
        img_rgb = np.stack([img.squeeze()] * 3, axis=-1)
    else:
        img_rgb = img
    
    superimposed = img_rgb * (1 - alpha) + heatmap_colored * alpha
    superimposed = np.clip(superimposed, 0, 1)
    
    ax.imshow(superimposed)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.axis('off')

print("Fonctions Grad-CAM définies ✅")

In [ ]:
# ============================================================
# 10.2 — Trouver la dernière couche convolutive
# ============================================================
last_conv_name = None
for layer in improved_model.layers:
    if isinstance(layer, layers.Conv2D):
        last_conv_name = layer.name

print(f"Dernière couche conv (CNN amélioré) : {last_conv_name}")

In [ ]:
# ============================================================
# 10.3 — Grad-CAM sur images bien classées, mal classées, ambiguës
# ============================================================
y_pred_improved = np.argmax(improved_model.predict(X_test, verbose=0), axis=1)
y_proba_improved = improved_model.predict(X_test, verbose=0)

correct_mask = (y_pred_improved == y_test)
incorrect_mask = ~correct_mask
max_confidence = np.max(y_proba_improved, axis=1)
ambiguous_mask = (max_confidence < 0.6) & correct_mask

n_examples = 3

def get_examples(mask, n):
    indices = np.where(mask)[0]
    if len(indices) == 0: return np.array([])
    return np.random.choice(indices, size=min(n, len(indices)), replace=False)

categories = [
    ("✅ Bien classées", get_examples(correct_mask, n_examples)),
    ("❌ Mal classées", get_examples(incorrect_mask, n_examples)),
    ("⚠️ Ambiguës (confiance < 60%)", get_examples(ambiguous_mask, n_examples))
]

for cat_name, indices in categories:
    if len(indices) == 0:
        print(f"\n{cat_name} : Aucun exemple trouvé")
        continue
    
    fig, axes = plt.subplots(len(indices), 3, figsize=(12, 4 * len(indices)))
    fig.suptitle(f"Grad-CAM — {cat_name}", fontsize=14, fontweight='bold')
    if len(indices) == 1: axes = axes[np.newaxis, :]
    
    for i, idx in enumerate(indices):
        img = X_test[idx]
        true_label = CLASSES[y_test[idx]]
        pred_label = CLASSES[y_pred_improved[idx]]
        conf = max_confidence[idx]
        
        axes[i, 0].imshow(img.squeeze(), cmap='gray')
        axes[i, 0].set_title(f"Vrai: {true_label}\nPréd: {pred_label} ({conf:.1%})")
        axes[i, 0].axis('off')
        
        heatmap = make_gradcam_heatmap(np.expand_dims(img, 0), improved_model, last_conv_name)
        
        axes[i, 1].imshow(heatmap, cmap='jet')
        axes[i, 1].set_title("Heatmap"); axes[i, 1].axis('off')
        
        display_gradcam(img, heatmap, title="Superposition", ax=axes[i, 2])
    
    plt.tight_layout(); plt.show()

 **Lecture des cartes Grad-CAM :**

**Sur les images bien classées :**
Les zones chaudes (rouges/jaunes) se concentrent sur les parties centrales du cerveau 
là où se trouvent les ventricules et les structures importantes.
C'est rassurant : le modèle ne regarde pas le fond noir ou les bords du crâne.

**Sur les images mal classées :**
Les activations sont plus dispersées, moins concentrées.
Le modèle "hésite"  il n'a pas trouvé de zone fortement discriminante.
C'est cohérent avec les confusions qu'on voit dans la matrice de confusion.

**Sur les images ambiguës (confiance < 0.6) :**
Les cartes sont encore plus étalées.
Quand le modèle n'est pas sûr, ses "regards" sont dispersés il cherche un signal mais ne le trouve pas nettement.

**Notre conclusion (en tant que data scientists, pas médecins) :**
Le modèle semble regarder des zones plausibles quand il est confiant.
Il ne semble pas décider sur du bruit aléatoire.
C'est le minimum qu'on peut vérifier sans expertise médicale.


### Limites de notre analyse Grad-CAM

Il faut être honnêtes sur ce qu'on peut et ne peut pas dire ici.

**Ce qu'on peut dire :**
- Le modèle regarde des zones internes du cerveau (pas les bords)
- Il est moins focalisé quand il se trompe
- Les activations sont plus diffuses sur les cas VeryMild  cohérent avec leur difficulté

**Ce qu'on ne peut PAS dire mais quon aimerai <3 :**
- "Le modèle a détecté une atrophie de l'hippocampe"
- "Il regarde la bonne zone anatomique"
- Toute interprétation médicale précise

Pour valider cliniquement ce que regarde le modèle, il faudrait un radiologue
qui compare les zones activées avec les vraies zones d'intérêt neurologiques.
Ce n'est pas notre rôle dans ce projet.


---
## 11. Augmentation par GAN DCGAN

### Le problème qu'on essaie de résoudre

La classe ModerateDemented a **64 images dans tout le dataset, 45 dans le train**.
Le déséquilibre avec NonDemented est de 50:1.

La data augmentation classique (rotations, zooms) aide un peu, mais ne crée pas
de vraie diversité  c'est toujours le même cerveau vu sous différents angles.

**Idée : et si on générais de nouvelles images synthétiques ?**
C'est le rôle d'un GAN (Generative Adversarial Network).

### Comment fonctionne un DCGAN (sans le jargon)

Le GAN c'est un jeu entre 2 réseaux :
- Le **générateur** essaie de créer de fausses images qui ressemblent à des vrais cerveaux
- Le **discriminateur** essaie de distinguer les vraies des fausses

Au fil de l'entraînement, les deux s'améliorent mutuellement.
Le générateur fait de meilleures fausses images, le discriminateur devient meilleur détecteur.
À la fin, on espère que le générateur peut créer des images assez réalistes pour tromper le classifieur.

### La limite qu'on anticipe

Un GAN entraîné sur **45 images**, c'est très peu.
En général, les GANs ont besoin de milliers d'exemples pour générer des images de qualité.
On va tester quand même, mais on anticipe que le résultat sera limité.


In [ ]:
# ============================================================
# 11.1 — Construction du DCGAN
# ============================================================
LATENT_DIM = 128
GAN_IMG_SIZE = 64

def build_generator(latent_dim=LATENT_DIM):
    model = models.Sequential(name='generator')
    model.add(layers.Dense(4 * 4 * 512, input_dim=latent_dim))
    model.add(layers.Reshape((4, 4, 512)))
    model.add(layers.BatchNormalization()); model.add(layers.ReLU())
    
    model.add(layers.Conv2DTranspose(256, (4,4), strides=(2,2), padding='same'))
    model.add(layers.BatchNormalization()); model.add(layers.ReLU())
    
    model.add(layers.Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
    model.add(layers.BatchNormalization()); model.add(layers.ReLU())
    
    model.add(layers.Conv2DTranspose(64, (4,4), strides=(2,2), padding='same'))
    model.add(layers.BatchNormalization()); model.add(layers.ReLU())
    
    model.add(layers.Conv2DTranspose(1, (4,4), strides=(2,2), padding='same', activation='tanh'))
    return model

def build_discriminator(img_shape=(GAN_IMG_SIZE, GAN_IMG_SIZE, 1)):
    model = models.Sequential(name='discriminator')
    model.add(layers.Conv2D(64, (4,4), strides=(2,2), padding='same', input_shape=img_shape))
    model.add(layers.LeakyReLU(0.2)); model.add(layers.Dropout(0.3))
    
    model.add(layers.Conv2D(128, (4,4), strides=(2,2), padding='same'))
    model.add(layers.BatchNormalization()); model.add(layers.LeakyReLU(0.2)); model.add(layers.Dropout(0.3))
    
    model.add(layers.Conv2D(256, (4,4), strides=(2,2), padding='same'))
    model.add(layers.BatchNormalization()); model.add(layers.LeakyReLU(0.2)); model.add(layers.Dropout(0.3))
    
    model.add(layers.Conv2D(512, (4,4), strides=(2,2), padding='same'))
    model.add(layers.BatchNormalization()); model.add(layers.LeakyReLU(0.2))
    
    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation='sigmoid'))
    return model

generator = build_generator()
discriminator = build_discriminator()
generator.summary()
print("\n")
discriminator.summary()

In [ ]:
# ============================================================
# 11.2 — Préparation des données GAN (classe ModerateDemented)
# ============================================================
discriminator.compile(
    optimizer=optimizers.Adam(learning_rate=2e-4, beta_1=0.5),
    loss='binary_crossentropy', metrics=['accuracy']
)

discriminator.trainable = False
gan_input = layers.Input(shape=(LATENT_DIM,))
generated_img = generator(gan_input)
gan_output = discriminator(generated_img)
gan = models.Model(gan_input, gan_output, name='dcgan')
gan.compile(optimizer=optimizers.Adam(learning_rate=2e-4, beta_1=0.5), loss='binary_crossentropy')

# Classe minoritaire = ModerateDemented (index 1)
minority_class = 1
minority_name = CLASSES[minority_class]
minority_images = X_train[y_train == minority_class]

# Redimensionner en 64x64 pour le GAN
minority_64 = np.array([cv2.resize(img.squeeze(), (GAN_IMG_SIZE, GAN_IMG_SIZE))
                         for img in minority_images])
minority_64 = np.expand_dims(minority_64, axis=-1)
minority_64 = minority_64 * 2.0 - 1.0  # Normaliser [-1, 1] pour tanh

print(f"Classe minoritaire : {minority_name}")
print(f"Images d'entraînement GAN : {minority_64.shape}")

 **Préparation du GAN :**

Le GAN s'entraîne sur les images ModerateDemented du **train uniquement** (45 images).
Important : il ne voit jamais les images du test  ce serait de la fuite d'information.

Les images sont redimensionnées en 64×64 (pas 224×224) pour le GAN.
Pourquoi ? Générer des images de 224×224 avec seulement 45 exemples serait encore plus difficile.
On génèrera ensuite en 64×64 et on upscalera pour le classifieur.

**Ce qu'on espère :** que le GAN apprend la structure générale d'une IRM ModerateDemented.
**Ce qu'on craint :** qu'avec 45 images, il reproduise surtout les mêmes patterns
et manque la diversité réelle de cette classe.


In [ ]:
# ============================================================
# 11.3 — Entraînement du DCGAN
# ============================================================
GAN_EPOCHS = 300
GAN_BATCH_SIZE = 16  # Petit batch car peu d'images
d_losses, g_losses = [], []

n_samples = len(minority_64)
steps_per_epoch = max(1, n_samples // GAN_BATCH_SIZE)

print(f"Entraînement DCGAN : {GAN_EPOCHS} epochs")
print(f"Images disponibles : {n_samples}")
print("=" * 50)

for epoch in range(GAN_EPOCHS):
    epoch_d_loss, epoch_g_loss = 0, 0
    
    for step in range(steps_per_epoch):
        # --- Discriminateur ---
        idx = np.random.randint(0, n_samples, GAN_BATCH_SIZE)
        real_imgs = minority_64[idx]
        
        noise = np.random.normal(0, 1, (GAN_BATCH_SIZE, LATENT_DIM))
        fake_imgs = generator.predict(noise, verbose=0)
        
        real_labels = np.ones((GAN_BATCH_SIZE, 1)) * 0.9
        fake_labels = np.zeros((GAN_BATCH_SIZE, 1)) + 0.1
        
        discriminator.trainable = True
        d_loss_real = discriminator.train_on_batch(real_imgs, real_labels)
        d_loss_fake = discriminator.train_on_batch(fake_imgs, fake_labels)
        d_loss = 0.5 * (d_loss_real[0] + d_loss_fake[0])
        
        # --- Générateur ---
        noise = np.random.normal(0, 1, (GAN_BATCH_SIZE, LATENT_DIM))
        discriminator.trainable = False
        g_loss = gan.train_on_batch(noise, np.ones((GAN_BATCH_SIZE, 1)))
        
        epoch_d_loss += d_loss
        epoch_g_loss += g_loss
    
    d_losses.append(epoch_d_loss / steps_per_epoch)
    g_losses.append(epoch_g_loss / steps_per_epoch)
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{GAN_EPOCHS} — D Loss: {d_losses[-1]:.4f} — G Loss: {g_losses[-1]:.4f}")

print("\nEntraînement DCGAN terminé ✅")

 **Lecture des losses du GAN :**

Un GAN s'entraîne bien quand les deux losses restent dans un équilibre dynamique.
Si le discriminateur devient trop fort, le générateur ne peut plus apprendre.
Si le générateur devient trop fort, le discriminateur ne peut plus s'améliorer.

Ce qu'on voit ici :
- Epoch 100 → D Loss: 0.36, G Loss: 0.04 (discriminateur très fort)
- Epoch 150 → D Loss: 0.89, G Loss: 1.47 (rééquilibrage)
- Epoch 300 → D Loss: 0.34, G Loss: 0.14 (relativement stable)

Ce comportement en dents de scie est classique et attendu pour un GAN.
Ça ne veut pas dire que les images générées sont bonnes  ça veut juste dire
que les deux réseaux se battent encore.

**La vraie question :** les images produites sont-elles assez plausibles pour aider le classifieur ?
La réponse sera dans la section 12.


In [ ]:
# ============================================================
# 11.4 — Visualiser les résultats du GAN
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(d_losses, label='Discriminator', alpha=0.7)
axes[0].plot(g_losses, label='Generator', alpha=0.7)
axes[0].set_title("Courbes de Loss DCGAN", fontweight='bold')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

noise = np.random.normal(0, 1, (16, LATENT_DIM))
generated = (generator.predict(noise, verbose=0) + 1) / 2.0

grid = np.zeros((4 * GAN_IMG_SIZE, 4 * GAN_IMG_SIZE))
for i in range(4):
    for j in range(4):
        grid[i*GAN_IMG_SIZE:(i+1)*GAN_IMG_SIZE,
             j*GAN_IMG_SIZE:(j+1)*GAN_IMG_SIZE] = generated[i*4+j].squeeze()

axes[1].imshow(grid, cmap='gray')
axes[1].set_title(f"Images synthétiques ({minority_name})", fontweight='bold')
axes[1].axis('off')

plt.tight_layout(); plt.show()

 **Les images générées par le GAN :**

Visuellement, les images ressemblent à des formes circulaires avec une structure centrale.
C'est approximativement la forme d'une coupe de cerveau  c'est déjà quelque chose.

**Mais soyons honnêtes :**
Avec seulement 45 exemples d'entraînement, le GAN n'a pas pu apprendre la vraie diversité
de la classe ModerateDemented. Les images générées sont probablement trop similaires entre elles
et manquent de certains détails subtils qui caractérisent vraiment ce stade.

**Ce qu'on va vérifier dans la section suivante :**
Est-ce que ces images imparfaites aident quand même le classifieur ?
Ou est-ce qu'elles l'embrouillent en lui apprenant de faux patterns ?


---
## 12. Réentraînement après enrichissement par GAN

### Le protocole de comparaison

Pour mesurer honnêtement l'impact du GAN, on compare :
- **CNN amélioré sans GAN** (section 7) notre référence
- **CNN amélioré avec GAN**  le même modèle, sur un train enrichi

**Le jeu de test ne change pas.** Seul le train change.
C'est la seule façon de mesurer l'impact réel du GAN.

**Les critères de succès :**
Si le GAN a aidé :
→ Le F1-macro monte ET le recall sur ModerateDemented reste bon

Si le GAN n'a pas aidé :
→ L'accuracy monte (faux positif), mais le F1-macro baisse ou le recall sur Moderate s'effondre

Un résultat négatif est aussi valable qu'un résultat positif en data science.
Il nous apprend quelque chose d'important sur les limites de l'approche.


In [ ]:
# ============================================================
# 12.1 — Générer et ajouter des images synthétiques
# ============================================================
class_counts_train = Counter(y_train)
majority_count = max(class_counts_train.values())
n_to_generate = majority_count - class_counts_train[minority_class]
print(f"Génération de {n_to_generate} images synthétiques pour {minority_name}")

all_synthetic = []
batch = 64
for i in range(0, n_to_generate, batch):
    n = min(batch, n_to_generate - i)
    noise = np.random.normal(0, 1, (n, LATENT_DIM))
    imgs = (generator.predict(noise, verbose=0) + 1) / 2.0
    all_synthetic.append(imgs)

synthetic_images = np.concatenate(all_synthetic, axis=0)[:n_to_generate]

# Redimensionner en 224x224
synthetic_224 = np.array([cv2.resize(img.squeeze(), (224, 224))
                          for img in synthetic_images])
synthetic_224 = np.expand_dims(synthetic_224, axis=-1)

X_train_enriched = np.concatenate([X_train, synthetic_224], axis=0)
y_train_enriched = np.concatenate([y_train, np.full(n_to_generate, minority_class)])
y_train_enriched_cat = to_categorical(y_train_enriched, NUM_CLASSES)

print(f"\nDataset enrichi : {len(X_train)} → {len(X_train_enriched)} images")
print(f"\nNouvelle distribution :")
for idx, cls in enumerate(CLASSES):
    count = np.sum(y_train_enriched == idx)
    print(f"  {cls}: {count}")

 **Dataset enrichi :**

| Classe | Avant GAN | Après GAN |
|:---|:---|:---|
| MildDemented | 627 | 627 |
| **ModerateDemented** | **45** | **2 240** |
| NonDemented | 2 240 | 2 240 |
| VeryMildDemented | 1 568 | 1 568 |
| **Total** | **4 480** | **6 675** |

On a ajouté **2 195 images synthétiques** pour ModerateDemented.
La classe passe de 45 à 2 240 exemples au niveau de NonDemented.

**Ce rééquilibrage est agressif.** Plus de 97% des images ModerateDemented dans le train
sont maintenant synthétiques. Si ces images ne sont pas assez réalistes,
le modèle va apprendre de faux patterns au lieu des vrais.


In [ ]:
# ============================================================
# 12.2 — Réentraîner le CNN amélioré
# ============================================================
improved_model_v2 = build_improved_cnn(INPUT_SHAPE, NUM_CLASSES)
improved_model_v2._name = 'improved_cnn_v2_gan'

improved_model_v2.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

train_gen_enriched = ImageDataGenerator(
    rotation_range=15, width_shift_range=0.1,
    height_shift_range=0.1, zoom_range=0.1,
    horizontal_flip=True, fill_mode='nearest'
).flow(X_train_enriched, y_train_enriched_cat, batch_size=BATCH_SIZE, seed=SEED)

cw_enriched = compute_class_weight('balanced', classes=np.unique(y_train_enriched), y=y_train_enriched)
cw_enriched_dict = dict(enumerate(cw_enriched))

v2_history = improved_model_v2.fit(
    train_gen_enriched,
    steps_per_epoch=len(X_train_enriched) // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=val_generator,
    validation_steps=len(X_val) // BATCH_SIZE,
    class_weight=cw_enriched_dict,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
    ],
    verbose=1
)

plot_training_history(v2_history, "CNN Amélioré + GAN")

**Le CNN amélioré se réentraîne sur le dataset enrichi...**

Ce qu'on surveille :
- Est-ce que la val_loss est stable ? (elle est très élevée au début → mauvais signe)
- Est-ce que le recall sur ModerateDemented reste bon ?
- Est-ce que le F1-macro s'améliore ou se dégrade ?

Les premières époques montrent une val_loss explosive (24.5 à epoch 1 !).
C'est inquiétant le modèle a du mal à généraliser sur les vraies images de validation
quand il a vu beaucoup d'images synthétiques imparfaites.


In [ ]:
# ============================================================
# 12.3 — Comparaison avant/après GAN
# ============================================================
v2_results = evaluate_model(improved_model_v2, X_test, y_test, y_test_cat, CLASSES, "CNN Amélioré + GAN")

print("\n" + "=" * 70)
print("  IMPACT DU GAN SUR LES PERFORMANCES")
print("=" * 70)

df_compare = pd.DataFrame({
    'Métrique': ['Accuracy', 'F1-Score (macro)', 'Precision (macro)', 'Recall (macro)'],
    'Sans GAN': [f"{improved_results['accuracy']:.4f}", f"{improved_results['f1_macro']:.4f}",
                 f"{improved_results['precision_macro']:.4f}", f"{improved_results['recall_macro']:.4f}"],
    'Avec GAN': [f"{v2_results['accuracy']:.4f}", f"{v2_results['f1_macro']:.4f}",
                 f"{v2_results['precision_macro']:.4f}", f"{v2_results['recall_macro']:.4f}"]
})
print(df_compare.to_string(index=False))

delta_f1 = v2_results['f1_macro'] - improved_results['f1_macro']
print(f"\nΔ F1-Score : {delta_f1:+.4f}")
if delta_f1 > 0:
    print("✅ Le GAN a amélioré les performances sur la classe minoritaire.")
else:
    print("⚠️ Le GAN n'a pas amélioré — images générées potentiellement pas assez réalistes.")

 **Impact du GAN  :**

| Métrique | Sans GAN | Avec GAN | Évolution |
|:---|:---|:---|:---|
| Accuracy | 51.98% | **59.79%** | +7.81% ✅ |
| F1-macro | **0.572** | 0.441 | **-0.131** ❌ |
| Recall ModerateDemented | 0.778 | **0.000** | **-78%** 🚨 |

**Le résultat est net : le GAN a amélioré l'accuracy mais dégradé tout le reste.**

**Pourquoi l'accuracy monte mais le F1-macro baisse ?**
Avec 2240 images synthétiques ModerateDemented, le modèle a appris à prédire souvent "Moderate"
sur certains types d'images. Sur le test, ces prédictions tombent souvent sur NonDemented ou VeryMild.
Résultat : ModerateDemented a un recall de 0% sur le test (0/9 cas détectés).

Le modèle a appris les patterns du GAN, pas les vrais patterns de ModerateDemented.

**Ce qu'on en retient :**
Générer beaucoup d'images avec un GAN entraîné sur 45 exemples ne fonctionne pas.
La quantité ne compense pas la mauvaise qualité.
Le CNN amélioré sans GAN reste notre meilleur modèle.


---
## 13. Critique

### 13.1 Synthèse des résultats 

| Modèle | Verdict |
|:---|:---|
| **Baseline CNN** | 50% d'accuracy en prédisant presque tout "NonDemented". Utile comme référence, pas comme outil. |
| **CNN Amélioré** | **Notre meilleur modèle** — F1-macro 0.572, toutes les classes sont détectées. |
| **ResNet50** | Meilleure accuracy (54.8%) mais moins équilibré. Intéressant si on veut ne rater aucun cas grave. |
| **CNN + GAN** | Accuracy artificielle (59.8%) mais ModerateDemented tombé à 0% recall. Le GAN a nui. |

### 13.2 Nos vraies limites 

| Limite | Impact réel |
|:---|:---|
| **Dataset kaggle déjà prétraité** | On ne sait pas exactement comment les images ont été préparées |
| **Seulement 64 images ModerateDemented** | Le GAN ne peut pas faire de miracle avec si peu |
| **Classification 2D** | Une coupe ne donne pas toute l'information d'un volume cérébral 3D |
| **VeryMild toujours difficile** | Recall max 0.33  la frontière clinique est vraiment floue |
| **Pas de validation externe** | Nos résultats sont sur ce dataset uniquement généralisation inconnue |

### 13.3 La zone grise : VeryMildDemented

C'est la classe la plus importante cliniquement (détection précoce) et la plus difficile pour nos modèles.
Aucun modèle ne dépasse 0.33 en recall sur cette classe.

Si on devait utiliser un de ces modèles en vrai, c'est sur cette classe qu'on ferait le plus d'erreurs.
Et c'est précisément les erreurs les plus coûteuses  rater un patient en stade VeryMild,
c'est rater la fenêtre d'intervention.

### 13.4 Ce projet n'est pas un outil médical

On le dit clairement : ce notebook est un projet pédagogique de data science.
Pour un vrai déploiement clinique, il faudrait :
- Des milliers de vraies images par classe
- Des tests sur plusieurs populations différentes
- Une validation par des radiologues
- Un cadre réglementaire médical 

C'est juste un passion qu'on a mis en exerge


---
## 14. Conclusion & Perspectives

### Ce qu'on a construit

Une pipeline complète de deep learning appliquée à un problème médical réel :
1. **Chargement et analyse** d'un dataset déséquilibré (50:1)
2. **Prétraitement** adapté (class weights, augmentation justifiée)
3. **3 modèles comparés** : baseline → CNN amélioré → ResNet50
4. **Grad-CAM** pour regarder où le modèle regarde
5. **GAN** pour tester l'enrichissement synthétique
6. **Comparaison rigoureuse** avec les bonnes métriques (F1-macro, pas juste accuracy)

### Ce qu'on a appris  les vraies leçons

**L'accuracy ne suffit pas.**
Un modèle à 50% d'accuracy qui prédit tout "NonDemented" n'est pas meilleur
qu'un modèle à 52% qui reconnaît les 4 classes.

**Plus de données synthétiques ≠ meilleur modèle.**
Le GAN entraîné sur 45 images a empiré les choses. Qualité > quantité.

**Le Transfer Learning n'est pas magique.**
ResNet50 pré-entraîné sur des photos naturelles apporte de l'accuracy,
mais pas forcément du F1-macro sur des IRM. Il faut mesurer, pas supposer.

**VeryMild est la frontière la plus difficile.**
Et c'est aussi la plus importante cliniquement. C'est là où l'IA aurait le plus de valeur
et là où elle a le plus de mal. Le problème dur est souvent le plus intéressant.

### Perspectives  ce qu'on ferait avec plus de temps

- **Plus de données réelles** : télécharger tous les disques OASIS pour avoir 198 sujets au lieu de 64
- **CNN 3D** : travailler sur les volumes IRM complets, pas juste une coupe 2D
- **Focal loss** : une fonction de perte conçue pour les datasets très déséquilibrés
- **Modèle multimodal** : combiner l'IRM avec les variables cliniques (âge, score MMSE...)
- **GAN conditionnel** avec de vraies données : générer des images par classe avec bien plus d'exemples

### Notre vision finale

On n'est pas des médecins. Mais après ce projet, on comprend mieux comment l'IA peut aider
à lire des données médicales complexes. Et on comprend aussi ses limites.

L'IA ne diagnostique pas Alzheimer. Elle aide à trier, à prioriser, à détecter plus tôt.
Dans un monde où les ressources médicales sont limitées et où le vieillissement est universel,
c'est déjà quelque chose d'important.

Le vieillissement est une descente de gradient pour tout le monde.
Mais avec les bons outils, on peut peut-être en ralentir la pente pour certains.


---
## Annexes

### A. Environnement technique


In [ ]:
# ============================================================
# Annexe — Informations système
# ============================================================
import sys
print(f"Python       : {sys.version.split()[0]}")
print(f"TensorFlow   : {tf.__version__}")
print(f"Keras        : 2.15.0")
print(f"NumPy        : {np.__version__}")
print(f"Scikit-learn : {__import__('sklearn').__version__}")
print(f"GPU          : {tf.config.list_physical_devices('GPU')}")
print(f"Seed         : {SEED}")

### B. Références

1. **ADNI** — Mueller, S.G. et al. (2005). *The Alzheimer's Disease Neuroimaging Initiative.* Neuroimaging Clinics of North America.
2. **OASIS** — Marcus, D.S. et al. (2007). *Open Access Series of Imaging Studies.* Journal of Cognitive Neuroscience.
3. **ResNet** — He, K. et al. (2016). *Deep Residual Learning for Image Recognition.* CVPR.
4. **Grad-CAM** — Selvaraju, R.R. et al. (2017). *Grad-CAM: Visual Explanations from Deep Networks.* ICCV.
5. **DCGAN** — Radford, A. et al. (2016). *Unsupervised Representation Learning with DCGANs.* ICLR.
6. **Alzheimer & DL** — Jo, T. et al. (2019). *Deep Learning in Alzheimer's Disease.* Frontiers in Aging Neuroscience.
7. **Dataset** — Alzheimer MRI Dataset, disponible sur Kaggle.
